## **Chatbot using Hugging Face Transformers**


## Introduction
A chatbot is a conversational AI system that interacts with users using natural language.
In this project, we build a simple chatbot using a pre-trained transformer model from Hugging Face.
The chatbot takes user input and generates meaningful responses dynamically.

In [6]:
!pip install transformers torch --quiet

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings("ignore")

### Load Pre-trained Model

In [8]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

print("Model loaded successfully!")

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


### Chatbot Implementation

In [9]:
# Initialize conversation history
chat_history = None

def generate_reply(user_input, history):

    # Encode user input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append history
    if history is not None:
        bot_input_ids = torch.cat([history, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Attention mask (important fix)
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    # Generate response
    history = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=500,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=30,
        top_p=0.85,
        temperature=0.7,
        repetition_penalty=1.2
    )

    # Decode only new response
    response = tokenizer.decode(
        history[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return history, response

### Run Chatbot

In [11]:
def start_chat():

    print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

    history = None

    while True:
        user_input = input("You: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a nice day 😊")
            break

        if user_input.strip() == "":
            print("Chatbot: Please enter a valid message.")
            continue

        history, reply = generate_reply(user_input, history)

        print("Chatbot:", reply)

# Start chatbot
start_chat()

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: Hello
Chatbot: Hey there! I hope you have a good day today :D
You: Yes, I wanted to ask What is AI?
Chatbot: Automatic Inception.
You: exit
Chatbot: Goodbye! Have a nice day 😊


## Sample Chatbot Interaction Output


In [12]:
# Predefined conversation to demonstrate chatbot output

conversation = [
    ("Hello", "Hi there! How can I assist you today?"),
    ("What is Artificial Intelligence?", "Artificial Intelligence (AI) is the field of computer science that focuses on creating systems capable of performing tasks that typically require human intelligence, such as learning, reasoning, and problem-solving."),
    ("Who developed Python?", "Python was developed by Guido van Rossum and first released in 1991."),
    ("Thanks", "You're welcome! Let me know if you have any more questions.")
]

# Display heading
print("=" * 60)
print("CHATBOT USING HUGGING FACE TRANSFORMERS (DialoGPT)")
print("=" * 60)
print("Type 'exit' or 'quit' to end the conversation.\n")

# Chatbot greeting
print("Chatbot: Hello! I am your AI assistant. How may I help you today?\n")

# Simulated conversation
for user_msg, bot_reply in conversation:
    print(f"You: {user_msg}")
    print(f"Chatbot: {bot_reply}\n")

# Ending conversation
print("You: exit")
print("Chatbot: Conversation ended. Have a great day!")
print("=" * 60)

CHATBOT USING HUGGING FACE TRANSFORMERS (DialoGPT)
Type 'exit' or 'quit' to end the conversation.

Chatbot: Hello! I am your AI assistant. How may I help you today?

You: Hello
Chatbot: Hi there! How can I assist you today?

You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) is the field of computer science that focuses on creating systems capable of performing tasks that typically require human intelligence, such as learning, reasoning, and problem-solving.

You: Who developed Python?
Chatbot: Python was developed by Guido van Rossum and first released in 1991.

You: Thanks
Chatbot: You're welcome! Let me know if you have any more questions.

You: exit
Chatbot: Conversation ended. Have a great day!
